In [1]:
# Import libaries
import pandas as pd
from sklearn.model_selection import train_test_split 
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest,f_regression
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
import pickle
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
dataset = pd.read_csv("synthetic_ev_battery_health.csv")

In [3]:
dataset

,Battery_ID,Cycles,Voltage_V,Current_A,Temperature_C,Internal_Resistance_mOhm,Capacity_Ah,SOH
0,B00001,165,3.43,3.45,29.2,40.2,47.54,95.33
1,B00002,386,4.19,3.09,30.0,48.6,46.98,90.41
2,B00003,303,4.18,3.96,28.8,39.6,46.37,91.79
3,B00004,1572,4.02,1.46,22.8,92.2,33.49,69.18
4,B00005,1708,3.30,0.89,27.7,100.9,33.65,66.52
...,...,...,...,...,...,...,...,...
9995,B09996,1076,3.82,1.18,25.5,75.0,39.35,77.71
9996,B09997,398,3.80,3.36,26.0,43.0,46.16,92.31
9997,B09998,452,3.64,4.98,28.7,47.2,46.04,89.68
9998,B09999,116,4.12,3.12,27.9,30.5,48.63,96.10


In [4]:
dataset.isnull().sum()

Battery_ID                  0
Cycles                      0
Voltage_V                   0
Current_A                   0
Temperature_C               0
Internal_Resistance_mOhm    0
Capacity_Ah                 0
SOH                         0
dtype: int64

In [5]:
dataset.describe()
    

,Cycles,Voltage_V,Current_A,Temperature_C,Internal_Resistance_mOhm,Capacity_Ah,SOH
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,999.763400,3.701544,2.741918,34.921090,70.022450,40.004203,78.502644
std,574.884249,0.289164,1.303848,8.631356,23.133155,5.771934,11.618488
min,0.000000,3.200000,0.500000,20.000000,25.200000,29.040000,55.100000
25%,504.000000,3.450000,1.620000,27.500000,50.300000,35.050000,68.560000
50%,999.000000,3.700000,2.750000,34.800000,69.950000,40.020000,78.540000
75%,1497.000000,3.950000,3.870000,42.300000,89.900000,44.942500,88.420000
max,2000.000000,4.200000,5.000000,50.000000,114.600000,50.940000,100.000000


In [6]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Battery_ID                10000 non-null  object 
 1   Cycles                    10000 non-null  int64  
 2   Voltage_V                 10000 non-null  float64
 3   Current_A                 10000 non-null  float64
 4   Temperature_C             10000 non-null  float64
 5   Internal_Resistance_mOhm  10000 non-null  float64
 6   Capacity_Ah               10000 non-null  float64
 7   SOH                       10000 non-null  float64
dtypes: float64(6), int64(1), object(1)
memory usage: 625.1+ KB


In [7]:
dataset.columns

Index(['Battery_ID', 'Cycles', 'Voltage_V', 'Current_A', 'Temperature_C',
       'Internal_Resistance_mOhm', 'Capacity_Ah', 'SOH'],
      dtype='object')

In [8]:
dataset.sample()

,Battery_ID,Cycles,Voltage_V,Current_A,Temperature_C,Internal_Resistance_mOhm,Capacity_Ah,SOH
6621,B06622,300,3.97,4.23,48.8,37.5,46.78,88.47


In [9]:
useless_col = ['Battery_ID']
dataset = dataset.drop(columns = useless_col, axis=1)

In [10]:
dataset.head()

,Cycles,Voltage_V,Current_A,Temperature_C,Internal_Resistance_mOhm,Capacity_Ah,SOH
0,165,3.43,3.45,29.2,40.2,47.54,95.33
1,386,4.19,3.09,30.0,48.6,46.98,90.41
2,303,4.18,3.96,28.8,39.6,46.37,91.79
3,1572,4.02,1.46,22.8,92.2,33.49,69.18
4,1708,3.30,0.89,27.7,100.9,33.65,66.52


In [11]:
df2 = pd.get_dummies(dataset, drop_first=True)

indep_x=df2.drop('SOH', axis=1)
dep_y=df2['SOH']

In [12]:
indep_x

,Cycles,Voltage_V,Current_A,Temperature_C,Internal_Resistance_mOhm,Capacity_Ah
0,165,3.43,3.45,29.2,40.2,47.54
1,386,4.19,3.09,30.0,48.6,46.98
2,303,4.18,3.96,28.8,39.6,46.37
3,1572,4.02,1.46,22.8,92.2,33.49
4,1708,3.30,0.89,27.7,100.9,33.65
...,...,...,...,...,...,...
9995,1076,3.82,1.18,25.5,75.0,39.35
9996,398,3.80,3.36,26.0,43.0,46.16
9997,452,3.64,4.98,28.7,47.2,46.04
9998,116,4.12,3.12,27.9,30.5,48.63


In [13]:
x_train, x_test, y_train, y_test = train_test_split(indep_x, dep_y, test_size = 0.30, random_state = 0)
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)



In [14]:
# for finding Feature Importance
from sklearn.feature_selection import f_regression
regressor = LinearRegression()
regressor.fit(x_train, y_train)
test = SelectKBest(score_func = f_regression, k = 2)
fit1 = test.fit(indep_x, dep_y)
selectk_features = fit1.transform(indep_x)
feature_scores = fit1.scores_
top_indices = fit1.get_support(indices = True)
feature_names = indep_x.columns
weight = regressor.coef_
bias = regressor.intercept_

In [15]:
y_pred = regressor.predict(x_test)

In [16]:
y_pred

array([94.0206703 , 75.52069862, 59.02538823, ..., 74.43751788,
       80.68831064, 68.06904727])

In [17]:
from sklearn.metrics import r2_score
r_score = r2_score(y_test, y_pred)
r_score

0.9901105110060545

In [18]:
# Save model
import pickle
filename = 'SOH_Final_Model.sav'
pickle.dump(regressor,open(filename,'wb'))
load_model = pickle.load(open(filename,'rb'))

In [19]:
dataset.head()

,Cycles,Voltage_V,Current_A,Temperature_C,Internal_Resistance_mOhm,Capacity_Ah,SOH
0,165,3.43,3.45,29.2,40.2,47.54,95.33
1,386,4.19,3.09,30.0,48.6,46.98,90.41
2,303,4.18,3.96,28.8,39.6,46.37,91.79
3,1572,4.02,1.46,22.8,92.2,33.49,69.18
4,1708,3.30,0.89,27.7,100.9,33.65,66.52


In [20]:
data = {
    'Cycles': int(input('Cycles (165): ')),
    'Voltage_V': float(input('Voltage_V (3.43): ')),
    'Current_A': float(input('Current_A (3.45): ')),
    'Temperature_C': float(input('Temperature_C (29.2): ')),
    'Internal_Resistance_mOhm': float(input('Internal_Resistance_mOhm (40.2):')),
    'Capacity_Ah': float(input('Capacity_Ah (47.5): '))
}

Cycles (165):  165
Voltage_V (3.43):  3.43
Current_A (3.45):  3.43
Temperature_C (29.2):  29.2
Internal_Resistance_mOhm (40.2): 40.2
Capacity_Ah (47.5):  47.2


In [24]:
df = pd.DataFrame([data])
preinput = sc.transform(df)
result = load_model.predict(preinput)

In [25]:
result

array([96.06559753])

In [26]:
print("\nBattery Health:", "Healthy Battery" if result[0] >= 80 else "Degraded")


Battery Health: Healthy Battery
